In [1]:
%pip install -q transformers accelerate bitsandbytes
%pip install -q peft trl datasets
%pip install -q pandas numpy scikit-learn tqdm jiwer

In [2]:
import os
import json
import gc
import time
import warnings
import random

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
PROJECT_ROOT = "/content/drive/MyDrive/BharatDocAI"

ANNOTATION_PATH = os.path.join(
    PROJECT_ROOT,
    "data/annotations/synthetic_documents.jsonl"
)

MANIFEST_PATH = os.path.join(
    PROJECT_ROOT,
    "data/processed/degradation_manifest.csv"
)

TESSERACT_DIR = os.path.join(
    PROJECT_ROOT,
    "data/ocr_outputs/tesseract"
)

EASYOCR_RESULTS_PATH = os.path.join(
    PROJECT_ROOT,
    "data/ocr_outputs/easyocr/easyocr_all_results.jsonl"
)

PADDLEOCR_RESULTS_PATH = os.path.join(
    PROJECT_ROOT,
    "data/ocr_outputs/paddleocr/paddleocr_all_results.jsonl"
)

OUTPUT_DIR = os.path.join(
    PROJECT_ROOT,
    "experiments/baselines/llm_extraction"
)

METRICS_DIR = os.path.join(
    PROJECT_ROOT,
    "outputs/metrics/llm_extraction"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

print("Project paths configured!")

Project paths configured!


In [5]:
required_files = [
    ANNOTATION_PATH,
    MANIFEST_PATH,
    os.path.join(
        TESSERACT_DIR,
        "tesseract_clean_results.jsonl"
    ),
    os.path.join(
        TESSERACT_DIR,
        "tesseract_degraded_results.jsonl"
    ),
    EASYOCR_RESULTS_PATH,
    PADDLEOCR_RESULTS_PATH
]

for path in required_files:

    status = os.path.exists(path)

    print(
        "[OK]" if status else "[MISSING]",
        path
    )

assert all(os.path.exists(p) for p in required_files)

print("\nAll required datasets are available!")

[OK] /content/drive/MyDrive/BharatDocAI/data/annotations/synthetic_documents.jsonl
[OK] /content/drive/MyDrive/BharatDocAI/data/processed/degradation_manifest.csv
[OK] /content/drive/MyDrive/BharatDocAI/data/ocr_outputs/tesseract/tesseract_clean_results.jsonl
[OK] /content/drive/MyDrive/BharatDocAI/data/ocr_outputs/tesseract/tesseract_degraded_results.jsonl
[OK] /content/drive/MyDrive/BharatDocAI/data/ocr_outputs/easyocr/easyocr_all_results.jsonl
[OK] /content/drive/MyDrive/BharatDocAI/data/ocr_outputs/paddleocr/paddleocr_all_results.jsonl

All required datasets are available!


In [6]:
# Load synthetic document annotations

with open(ANNOTATION_PATH, "r", encoding="utf-8") as f:
    annotations = [
        json.loads(line)
        for line in f
        if line.strip()
    ]


# Load Tesseract results

def load_jsonl(path):

    with open(path, "r", encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]


tesseract_clean = load_jsonl(
    os.path.join(
        TESSERACT_DIR,
        "tesseract_clean_results.jsonl"
    )
)

tesseract_degraded = load_jsonl(
    os.path.join(
        TESSERACT_DIR,
        "tesseract_degraded_results.jsonl"
    )
)


# Load EasyOCR and PaddleOCR results

easyocr_results = load_jsonl(
    EASYOCR_RESULTS_PATH
)

paddleocr_results = load_jsonl(
    PADDLEOCR_RESULTS_PATH
)


print("DATASET COUNTS")
print("-" * 45)

print("Annotations:", len(annotations))
print("Tesseract clean:", len(tesseract_clean))
print("Tesseract degraded:", len(tesseract_degraded))
print("EasyOCR:", len(easyocr_results))
print("PaddleOCR:", len(paddleocr_results))

DATASET COUNTS
---------------------------------------------
Annotations: 90
Tesseract clean: 90
Tesseract degraded: 1350
EasyOCR: 1440
PaddleOCR: 1440


In [7]:
print("ANNOTATION KEYS:")
print(annotations[0].keys())

print("\nFIRST ANNOTATION:")
print(
    json.dumps(
        annotations[0],
        indent=4,
        ensure_ascii=False
    )
)

print("\nDOCUMENT CATEGORIES:")

print(
    pd.Series([
        item.get("category")
        for item in annotations
    ]).value_counts()
)

print("\nLANGUAGES:")

print(
    pd.Series([
        item.get("language")
        for item in annotations
    ]).value_counts()
)

ANNOTATION KEYS:
dict_keys(['document_id', 'language', 'category', 'template_family', 'split', 'image_path', 'ground_truth', 'document_text'])

FIRST ANNOTATION:
{
    "document_id": "english_invoice_001",
    "language": "english",
    "category": "invoice",
    "template_family": "T1",
    "split": "train",
    "image_path": "/content/drive/MyDrive/BharatDocAI/data/raw_documents/english_invoice_001.png",
    "ground_truth": {
        "name": "Kiran Rao",
        "date": "01/01/2026",
        "document_id": "INV-0001",
        "address": "Bengaluru",
        "amount": "1637"
    },
    "document_text": "INVOICE\nCustomer Name: Kiran Rao\nInvoice Date: 01/01/2026\nInvoice Number: INV-0001\nAddress: Bengaluru\nTotal Amount: 1637"
}

DOCUMENT CATEGORIES:
invoice        30
application    30
certificate    30
Name: count, dtype: int64

LANGUAGES:
english    30
hindi      30
kannada    30
Name: count, dtype: int64


In [8]:
print("TESSERACT RESULT KEYS:")
print(tesseract_clean[0].keys())

print("\nTESSERACT SAMPLE:")
print(
    json.dumps(
        tesseract_clean[0],
        indent=4,
        ensure_ascii=False
    )
)

print("\nEASYOCR RESULT KEYS:")
print(easyocr_results[0].keys())

print("\nPADDLEOCR RESULT KEYS:")
print(paddleocr_results[0].keys())

TESSERACT RESULT KEYS:
dict_keys(['document_id', 'language', 'category', 'split', 'corruption', 'severity', 'image_path', 'ground_truth', 'ocr_text', 'status'])

TESSERACT SAMPLE:
{
    "document_id": "english_invoice_001",
    "language": "english",
    "category": "invoice",
    "split": "train",
    "corruption": "clean",
    "severity": "none",
    "image_path": "/content/drive/MyDrive/BharatDocAI/data/raw_documents/english_invoice_001.png",
    "ground_truth": "INVOICE\nCustomer Name: Kiran Rao\nInvoice Date: 01/01/2026\nInvoice Number: INV-0001\nAddress: Bengaluru\nTotal Amount: 1637",
    "ocr_text": "INVOICE\n\nCustomer Name:\n\nInvoice Date:\n\nInvoice Number:\n\nAddress:\n\nTotal Amount:\n\nKiran Rao\n\n01/01/2026\n\nINV-0001\n\nBengaluru\n\n1637",
    "status": "success"
}

EASYOCR RESULT KEYS:
dict_keys(['engine', 'document_id', 'language', 'category', 'split', 'corruption', 'severity', 'parameter', 'image_path', 'ground_truth', 'ocr_text', 'status'])

PADDLEOCR RESULT KEYS

In [9]:
manifest_df = pd.read_csv(MANIFEST_PATH)

print("DOCUMENT SPLIT DISTRIBUTION")
print(
    manifest_df[
        manifest_df["corruption"] == "clean"
    ]["split"].value_counts()
)

print("\nLANGUAGE-WISE SPLIT DISTRIBUTION")

display(
    manifest_df[
        manifest_df["corruption"] == "clean"
    ].pivot_table(
        index="language",
        columns="split",
        values="document_id",
        aggfunc="count"
    )
)

DOCUMENT SPLIT DISTRIBUTION
split
train         54
validation    18
test          18
Name: count, dtype: int64

LANGUAGE-WISE SPLIT DISTRIBUTION


split,test,train,validation
language,,,
english,6,18,6
hindi,6,18,6
kannada,6,18,6


In [10]:
from collections import defaultdict

category_fields = defaultdict(set)

for item in annotations:

    category = item["category"]

    fields = item["ground_truth"].keys()

    category_fields[category].update(fields)


print("GROUND-TRUTH FIELDS BY DOCUMENT CATEGORY")
print("=" * 60)

for category, fields in category_fields.items():

    print(f"\n{category.upper()}")

    for field in sorted(fields):
        print(" -", field)

GROUND-TRUTH FIELDS BY DOCUMENT CATEGORY

INVOICE
 - address
 - amount
 - date
 - document_id
 - name

APPLICATION
 - address
 - date
 - document_id
 - name
 - phone

CERTIFICATE
 - date
 - document_id
 - grade
 - institution
 - name


In [11]:
for category in ["invoice", "application", "certificate"]:

    print("\n" + "=" * 70)
    print("CATEGORY:", category.upper())

    sample = next(
        item for item in annotations
        if item["category"] == category
    )

    print("\nDocument ID:", sample["document_id"])
    print("Language:", sample["language"])

    print("\nStructured ground truth:")

    print(
        json.dumps(
            sample["ground_truth"],
            indent=4,
            ensure_ascii=False
        )
    )

    print("\nOriginal document text:")
    print(sample["document_text"])


CATEGORY: INVOICE

Document ID: english_invoice_001
Language: english

Structured ground truth:
{
    "name": "Kiran Rao",
    "date": "01/01/2026",
    "document_id": "INV-0001",
    "address": "Bengaluru",
    "amount": "1637"
}

Original document text:
INVOICE
Customer Name: Kiran Rao
Invoice Date: 01/01/2026
Invoice Number: INV-0001
Address: Bengaluru
Total Amount: 1637

CATEGORY: APPLICATION

Document ID: english_application_011
Language: english

Structured ground truth:
{
    "name": "Sneha Nair",
    "date": "11/01/2026",
    "document_id": "APP-0011",
    "address": "Bengaluru",
    "phone": "9000001507"
}

Original document text:
APPLICATION FORM
Applicant Name: Sneha Nair
Application Date: 11/01/2026
Application ID: APP-0011
Address: Bengaluru
Phone Number: 9000001507

CATEGORY: CERTIFICATE

Document ID: english_certificate_021
Language: english

Structured ground truth:
{
    "name": "Priya Kumar",
    "date": "21/01/2026",
    "document_id": "CERT-0021",
    "institution"

In [12]:
annotation_lookup = {
    item["document_id"]: item
    for item in annotations
}

all_ocr_results = (
    tesseract_clean
    + tesseract_degraded
    + easyocr_results
    + paddleocr_results
)

print("Total OCR records:", len(all_ocr_results))

unknown_documents = []

metadata_mismatches = []

for result in all_ocr_results:

    doc_id = result["document_id"]

    if doc_id not in annotation_lookup:
        unknown_documents.append(doc_id)
        continue

    annotation = annotation_lookup[doc_id]

    for field in [
        "language",
        "category",
        "split"
    ]:

        if result[field] != annotation[field]:

            metadata_mismatches.append({
                "document_id": doc_id,
                "field": field,
                "ocr_value": result[field],
                "annotation_value": annotation[field]
            })


print("Unknown documents:", len(unknown_documents))
print("Metadata mismatches:", len(metadata_mismatches))

assert len(unknown_documents) == 0
assert len(metadata_mismatches) == 0

print("\nOCR-to-annotation alignment verified!")

Total OCR records: 4320
Unknown documents: 0
Metadata mismatches: 0

OCR-to-annotation alignment verified!


In [13]:
# Create a lookup using the top-level document ID.
annotation_lookup = {
    item["document_id"]: item
    for item in annotations
}

print("Annotation lookup created:", len(annotation_lookup))

# Define the exact schemas discovered from your annotations.
CATEGORY_SCHEMAS = {
    "invoice": [
        "name",
        "date",
        "document_id",
        "address",
        "amount"
    ],
    "application": [
        "name",
        "date",
        "document_id",
        "address",
        "phone"
    ],
    "certificate": [
        "name",
        "date",
        "document_id",
        "institution",
        "grade"
    ]
}

print("\nCategory schemas:")
for category, fields in CATEGORY_SCHEMAS.items():
    print(f"{category}: {fields}")

Annotation lookup created: 90

Category schemas:
invoice: ['name', 'date', 'document_id', 'address', 'amount']
application: ['name', 'date', 'document_id', 'address', 'phone']
certificate: ['name', 'date', 'document_id', 'institution', 'grade']


In [14]:
# Combine the three OCR result collections.
ocr_sources = {
    "tesseract": tesseract_clean + tesseract_degraded,
    "easyocr": easyocr_results,
    "paddleocr": paddleocr_results
}

unified_records = []

for engine, records in ocr_sources.items():

    for record in tqdm(records, desc=f"Processing {engine}"):

        doc_id = record["document_id"]

        # Retrieve the original annotation.
        annotation = annotation_lookup[doc_id]

        category = annotation["category"]

        # Get category-specific ground truth.
        full_ground_truth = annotation["ground_truth"]

        structured_ground_truth = {
            field: str(full_ground_truth.get(field, ""))
            for field in CATEGORY_SCHEMAS[category]
        }

        unified_record = {
            "document_id": doc_id,
            "language": annotation["language"],
            "category": category,
            "template_family": annotation["template_family"],
            "split": annotation["split"],

            "ocr_engine": engine,
            "corruption": record["corruption"],
            "severity": record["severity"],

            "ocr_text": record["ocr_text"],
            "ground_truth": structured_ground_truth,

            "ocr_status": record.get("status", "unknown")
        }

        unified_records.append(unified_record)

print("\nUnified dataset created!")
print("Total records:", len(unified_records))

Processing tesseract:   0%|          | 0/1440 [00:00<?, ?it/s]

Processing easyocr:   0%|          | 0/1440 [00:00<?, ?it/s]

Processing paddleocr:   0%|          | 0/1440 [00:00<?, ?it/s]


Unified dataset created!
Total records: 4320


In [15]:
unified_df = pd.DataFrame(unified_records)

print("Total records:", len(unified_df))

print("\nRecords by OCR engine:")
print(unified_df["ocr_engine"].value_counts())

print("\nRecords by split:")
print(unified_df["split"].value_counts())

print("\nRecords by language:")
print(unified_df["language"].value_counts())

print("\nRecords by category:")
print(unified_df["category"].value_counts())

# Verify expected record count.
assert len(unified_df) == 4320

# Verify that all three engines have 1440 records.
assert all(
    unified_df["ocr_engine"].value_counts()[engine] == 1440
    for engine in ["tesseract", "easyocr", "paddleocr"]
)

# Verify category-specific fields.
for _, row in unified_df.iterrows():

    expected_fields = set(CATEGORY_SCHEMAS[row["category"]])
    actual_fields = set(row["ground_truth"].keys())

    assert actual_fields == expected_fields, (
        f"Schema mismatch in {row['document_id']}"
    )

# Check duplicate records.
duplicate_count = unified_df.duplicated(
    subset=[
        "document_id",
        "ocr_engine",
        "corruption",
        "severity"
    ]
).sum()

print("\nDuplicate records:", duplicate_count)

assert duplicate_count == 0

print("\nUNIFIED DATASET VALIDATION PASSED!")

Total records: 4320

Records by OCR engine:
ocr_engine
tesseract    1440
easyocr      1440
paddleocr    1440
Name: count, dtype: int64

Records by split:
split
train         2592
validation     864
test           864
Name: count, dtype: int64

Records by language:
language
english    1440
hindi      1440
kannada    1440
Name: count, dtype: int64

Records by category:
category
invoice        1440
application    1440
certificate    1440
Name: count, dtype: int64

Duplicate records: 0

UNIFIED DATASET VALIDATION PASSED!


In [16]:
UNIFIED_JSONL_PATH = os.path.join(
    OUTPUT_DIR,
    "unified_extraction_dataset.jsonl"
)

UNIFIED_CSV_PATH = os.path.join(
    OUTPUT_DIR,
    "unified_extraction_dataset.csv"
)

with open(UNIFIED_JSONL_PATH, "w", encoding="utf-8") as f:

    for record in unified_records:
        f.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

unified_df.to_csv(
    UNIFIED_CSV_PATH,
    index=False,
    encoding="utf-8"
)

print("Saved unified dataset:")
print(UNIFIED_JSONL_PATH)
print(UNIFIED_CSV_PATH)

Saved unified dataset:
/content/drive/MyDrive/BharatDocAI/experiments/baselines/llm_extraction/unified_extraction_dataset.jsonl
/content/drive/MyDrive/BharatDocAI/experiments/baselines/llm_extraction/unified_extraction_dataset.csv


In [17]:
EXTRACTION_INSTRUCTIONS = {
    "invoice": """
Extract the following fields from the OCR text:
name, date, document_id, address, amount.

Return exactly these five fields.
""",

    "application": """
Extract the following fields from the OCR text:
name, date, document_id, address, phone.

Return exactly these five fields.
""",

    "certificate": """
Extract the following fields from the OCR text:
name, date, document_id, institution, grade.

Return exactly these five fields.
"""
}


def build_extraction_prompt(record):

    category = record["category"]
    instruction = EXTRACTION_INSTRUCTIONS[category]

    return f"""
You are an information extraction system.

Your task is to extract structured information from noisy OCR text.

Document category: {category}
Document language: {record['language']}

{instruction}

Rules:
1. Extract only information explicitly supported by the OCR text.
2. Do not invent or guess values.
3. If a field is missing or unreadable, return an empty string.
4. Preserve the original value as closely as possible.
5. Return valid JSON only.
6. Do not include explanations or Markdown.

OCR TEXT:
{record['ocr_text']}

Return a JSON object with exactly the required fields.
""".strip()

In [18]:
sample_records = unified_records[:3]

for record in sample_records:

    print("=" * 80)
    print("DOCUMENT:", record["document_id"])
    print("ENGINE:", record["ocr_engine"])
    print("CATEGORY:", record["category"])
    print("LANGUAGE:", record["language"])

    print("\nPROMPT:")
    print(build_extraction_prompt(record))

    print("\nEXPECTED OUTPUT:")
    print(json.dumps(
        record["ground_truth"],
        ensure_ascii=False,
        indent=4
    ))

DOCUMENT: english_invoice_001
ENGINE: tesseract
CATEGORY: invoice
LANGUAGE: english

PROMPT:
You are an information extraction system.

Your task is to extract structured information from noisy OCR text.

Document category: invoice
Document language: english


Extract the following fields from the OCR text:
name, date, document_id, address, amount.

Return exactly these five fields.


Rules:
1. Extract only information explicitly supported by the OCR text.
2. Do not invent or guess values.
3. If a field is missing or unreadable, return an empty string.
4. Preserve the original value as closely as possible.
5. Return valid JSON only.
6. Do not include explanations or Markdown.

OCR TEXT:
INVOICE

Customer Name:

Invoice Date:

Invoice Number:

Address:

Total Amount:

Kiran Rao

01/01/2026

INV-0001

Bengaluru

1637

Return a JSON object with exactly the required fields.

EXPECTED OUTPUT:
{
    "name": "Kiran Rao",
    "date": "01/01/2026",
    "document_id": "INV-0001",
    "address": 

In [19]:

EXTRACTION_INSTRUCTIONS = {
    "invoice": """
Extract these fields:
name, date, document_id, address, amount.
Return exactly these five fields.
""",

    "application": """
Extract these fields:
name, date, document_id, address, phone.
Return exactly these five fields.
""",

    "certificate": """
Extract these fields:
name, date, document_id, institution, grade.
Return exactly these five fields.
"""
}


def build_extraction_prompt(record):

    category = record["category"]
    instruction = EXTRACTION_INSTRUCTIONS[category]

    return f"""
You are an information extraction system.

Extract structured information from noisy OCR text.

Document category: {category}
Document language: {record['language']}

{instruction}

Rules:
1. Extract only information explicitly supported by the OCR text.
2. Never invent or guess values.
3. If a field is missing or unreadable, return an empty string.
4. Preserve original values as closely as possible.
5. Return valid JSON only.
6. Do not include explanations or Markdown.

OCR TEXT:
{record['ocr_text']}

Return a JSON object with exactly the required fields.
""".strip()


In [20]:

sample_records = unified_records[:3]

for record in sample_records:

    print("=" * 80)
    print("DOCUMENT:", record["document_id"])
    print("ENGINE:", record["ocr_engine"])
    print("CATEGORY:", record["category"])
    print("LANGUAGE:", record["language"])

    print("\nPROMPT:")
    print(build_extraction_prompt(record))

    print("\nEXPECTED OUTPUT:")
    print(json.dumps(
        record["ground_truth"],
        ensure_ascii=False,
        indent=4
    ))


DOCUMENT: english_invoice_001
ENGINE: tesseract
CATEGORY: invoice
LANGUAGE: english

PROMPT:
You are an information extraction system.

Extract structured information from noisy OCR text.

Document category: invoice
Document language: english


Extract these fields:
name, date, document_id, address, amount.
Return exactly these five fields.


Rules:
1. Extract only information explicitly supported by the OCR text.
2. Never invent or guess values.
3. If a field is missing or unreadable, return an empty string.
4. Preserve original values as closely as possible.
5. Return valid JSON only.
6. Do not include explanations or Markdown.

OCR TEXT:
INVOICE

Customer Name:

Invoice Date:

Invoice Number:

Address:

Total Amount:

Kiran Rao

01/01/2026

INV-0001

Bengaluru

1637

Return a JSON object with exactly the required fields.

EXPECTED OUTPUT:
{
    "name": "Kiran Rao",
    "date": "01/01/2026",
    "document_id": "INV-0001",
    "address": "Bengaluru",
    "amount": "1637"
}
DOCUMENT: e

In [21]:

# Select clean validation records from all three OCR engines.

zero_shot_validation_records = [
    record for record in unified_records
    if record["split"] == "validation"
    and record["corruption"] == "clean"
]

print("Total validation records:", len(zero_shot_validation_records))

print("\nRecords by OCR engine:")
print(
    pd.Series([
        record["ocr_engine"]
        for record in zero_shot_validation_records
    ]).value_counts()
)

print("\nRecords by language:")
print(
    pd.Series([
        record["language"]
        for record in zero_shot_validation_records
    ]).value_counts()
)

print("\nRecords by category:")
print(
    pd.Series([
        record["category"]
        for record in zero_shot_validation_records
    ]).value_counts()
)

assert len(zero_shot_validation_records) == 54

print("\nValidation sample selection successful!")


Total validation records: 54

Records by OCR engine:
tesseract    18
easyocr      18
paddleocr    18
Name: count, dtype: int64

Records by language:
english    18
hindi      18
kannada    18
Name: count, dtype: int64

Records by category:
invoice        18
application    18
certificate    18
Name: count, dtype: int64

Validation sample selection successful!


In [22]:

import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    gpu_properties = torch.cuda.get_device_properties(0)

    print(
        "Total GPU memory:",
        round(gpu_properties.total_memory / (1024**3), 2),
        "GB"
    )

    print(
        "Currently allocated:",
        round(torch.cuda.memory_allocated() / (1024**3), 2),
        "GB"
    )
else:
    print("Please reconnect using a GPU runtime.")


CUDA available: True
GPU: Tesla T4
Total GPU memory: 14.56 GB
Currently allocated: 0.0 GB


In [23]:

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "Qwen/Qwen3-4B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("4-bit quantization configuration created!")


4-bit quantization configuration created!


In [24]:

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

model.eval()

print("Qwen3-4B loaded successfully!")

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / (1024**3), 2),
    "GB"
)


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (3 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3-4B loaded successfully!
GPU memory allocated: 2.49 GB


In [25]:

def extract_with_qwen(record):

    prompt = build_extraction_prompt(record)

    messages = [
        {
            "role": "system",
            "content": "You are a precise structured information extraction system. Return only valid JSON."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()


In [26]:

test_record = zero_shot_validation_records[0]

print("DOCUMENT:", test_record["document_id"])
print("LANGUAGE:", test_record["language"])
print("CATEGORY:", test_record["category"])
print("OCR ENGINE:", test_record["ocr_engine"])

print("\nOCR TEXT:")
print(test_record["ocr_text"])

print("\nGROUND TRUTH:")
print(json.dumps(
    test_record["ground_truth"],
    ensure_ascii=False,
    indent=4
))

print("\nQWEN3 PREDICTION:")
test_prediction = extract_with_qwen(test_record)
print(test_prediction)


DOCUMENT: english_invoice_007
LANGUAGE: english
CATEGORY: invoice
OCR ENGINE: tesseract

OCR TEXT:
INVOICE

Customer Name : Ravi Sharma
Invoice Date : 07/01/2026
Invoice Number :  INV-0007
Address > Tumakuru

Total Amount : 2459

GROUND TRUTH:
{
    "name": "Ravi Sharma",
    "date": "07/01/2026",
    "document_id": "INV-0007",
    "address": "Tumakuru",
    "amount": "2459"
}

QWEN3 PREDICTION:
{
  "name": "Ravi Sharma",
  "date": "07/01/2026",
  "document_id": "INV-0007",
  "address": "Tumakuru",
  "amount": "2459"
}


In [27]:

import time

zero_shot_results = []

start_time = time.time()

for idx, record in enumerate(
    tqdm(zero_shot_validation_records, desc="Qwen3 Zero-Shot")
):

    prediction = extract_with_qwen(record)

    result = {
        "document_id": record["document_id"],
        "language": record["language"],
        "category": record["category"],
        "split": record["split"],
        "ocr_engine": record["ocr_engine"],
        "corruption": record["corruption"],
        "severity": record["severity"],

        "ground_truth": record["ground_truth"],
        "prediction_raw": prediction
    }

    zero_shot_results.append(result)

    # Save incrementally in case the runtime disconnects.
    if (idx + 1) % 5 == 0:
        with open(
            os.path.join(OUTPUT_DIR, "qwen3_zero_shot_partial.jsonl"),
            "w",
            encoding="utf-8"
        ) as f:
            for item in zero_shot_results:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

elapsed = time.time() - start_time

print("\nInference completed!")
print("Total records:", len(zero_shot_results))
print("Time taken:", round(elapsed / 60, 2), "minutes")


Qwen3 Zero-Shot:   0%|          | 0/54 [00:00<?, ?it/s]


Inference completed!
Total records: 54
Time taken: 6.07 minutes


In [28]:

def parse_prediction(raw_text, expected_fields):

    try:
        text = raw_text.strip()

        # Handle accidental Markdown code fences.
        if text.startswith("```"):
            text = text.replace("```json", "", 1)
            text = text.replace("```", "").strip()

        prediction = json.loads(text)

        if not isinstance(prediction, dict):
            return {}, False

        # Keep only expected fields and fill missing ones.
        cleaned = {
            field: str(prediction.get(field, "")).strip()
            for field in expected_fields
        }

        return cleaned, True

    except (json.JSONDecodeError, TypeError):
        return {}, False


for result in zero_shot_results:

    expected_fields = CATEGORY_SCHEMAS[result["category"]]

    parsed, valid = parse_prediction(
        result["prediction_raw"],
        expected_fields
    )

    result["prediction"] = parsed
    result["json_valid"] = valid

print("JSON parsing completed!")

print(
    "Valid JSON:",
    sum(r["json_valid"] for r in zero_shot_results)
)

print(
    "Invalid JSON:",
    sum(not r["json_valid"] for r in zero_shot_results)
)


JSON parsing completed!
Valid JSON: 54
Invalid JSON: 0


In [29]:

field_results = []

for result in zero_shot_results:

    for field in CATEGORY_SCHEMAS[result["category"]]:

        expected = str(result["ground_truth"].get(field, "")).strip()
        predicted = str(result["prediction"].get(field, "")).strip()

        field_results.append({
            "document_id": result["document_id"],
            "language": result["language"],
            "category": result["category"],
            "ocr_engine": result["ocr_engine"],
            "field": field,
            "expected": expected,
            "predicted": predicted,
            "correct": expected == predicted
        })


field_df = pd.DataFrame(field_results)

field_accuracy = field_df["correct"].mean() * 100

document_results = []

for result in zero_shot_results:

    fields = CATEGORY_SCHEMAS[result["category"]]

    exact_match = all(
        str(result["ground_truth"].get(field, "")).strip()
        ==
        str(result["prediction"].get(field, "")).strip()
        for field in fields
    )

    document_results.append({
        "document_id": result["document_id"],
        "language": result["language"],
        "category": result["category"],
        "ocr_engine": result["ocr_engine"],
        "exact_match": exact_match
    })


document_df = pd.DataFrame(document_results)

exact_match_accuracy = (
    document_df["exact_match"].mean() * 100
)

print("FIELD ACCURACY:", round(field_accuracy, 2), "%")

print(
    "EXACT DOCUMENT MATCH:",
    round(exact_match_accuracy, 2),
    "%"
)


FIELD ACCURACY: 70.74 %
EXACT DOCUMENT MATCH: 35.19 %


In [30]:

print("\nFIELD ACCURACY BY LANGUAGE")

language_metrics = (
    field_df.groupby("language")["correct"]
    .mean()
    .mul(100)
    .round(2)
)

print(language_metrics)


print("\nFIELD ACCURACY BY OCR ENGINE")

engine_metrics = (
    field_df.groupby("ocr_engine")["correct"]
    .mean()
    .mul(100)
    .round(2)
)

print(engine_metrics)


print("\nEXACT DOCUMENT MATCH BY LANGUAGE")

print(
    document_df.groupby("language")["exact_match"]
    .mean()
    .mul(100)
    .round(2)
)


print("\nEXACT DOCUMENT MATCH BY OCR ENGINE")

print(
    document_df.groupby("ocr_engine")["exact_match"]
    .mean()
    .mul(100)
    .round(2)
)



FIELD ACCURACY BY LANGUAGE
language
english    96.67
hindi      73.33
kannada    42.22
Name: correct, dtype: float64

FIELD ACCURACY BY OCR ENGINE
ocr_engine
easyocr      77.78
paddleocr    66.67
tesseract    67.78
Name: correct, dtype: float64

EXACT DOCUMENT MATCH BY LANGUAGE
language
english    83.33
hindi      22.22
kannada     0.00
Name: exact_match, dtype: float64

EXACT DOCUMENT MATCH BY OCR ENGINE
ocr_engine
easyocr      38.89
paddleocr    44.44
tesseract    22.22
Name: exact_match, dtype: float64


In [31]:

print("FIELD-WISE ACCURACY")
print("=" * 60)

field_accuracy_by_type = (
    field_df.groupby("field")["correct"]
    .agg(["sum", "count", "mean"])
)

field_accuracy_by_type["accuracy_percent"] = (
    field_accuracy_by_type["mean"] * 100
).round(2)

print(
    field_accuracy_by_type[
        ["sum", "count", "accuracy_percent"]
    ].to_string()
)


print("\nFIELD-WISE ACCURACY BY LANGUAGE")
print("=" * 60)

language_field_accuracy = (
    field_df.groupby(["language", "field"])["correct"]
    .mean()
    .mul(100)
    .round(2)
    .unstack()
)

print(language_field_accuracy.to_string())


FIELD-WISE ACCURACY
             sum  count  accuracy_percent
field                                    
address       30     36             83.33
amount        17     18             94.44
date          40     54             74.07
document_id   30     54             55.56
grade          8     18             44.44
institution   14     18             77.78
name          38     54             70.37
phone         14     18             77.78

FIELD-WISE ACCURACY BY LANGUAGE
field     address  amount    date  document_id  grade  institution    name   phone
language                                                                          
english    100.00  100.00  100.00       100.00  50.00       100.00  100.00  100.00
hindi       66.67  100.00   88.89        44.44  33.33        83.33   83.33  100.00
kannada     83.33   83.33   33.33        22.22  50.00        50.00   27.78   33.33


In [32]:

field_df["missing_prediction"] = (
    (field_df["predicted"] == "") &
    (field_df["expected"] != "")
)

field_df["incorrect_nonempty"] = (
    (field_df["predicted"] != "") &
    (field_df["predicted"] != field_df["expected"])
)

print("TOTAL FIELD PREDICTIONS:", len(field_df))

print(
    "Missing predictions:",
    int(field_df["missing_prediction"].sum())
)

print(
    "Incorrect nonempty predictions:",
    int(field_df["incorrect_nonempty"].sum())
)

print(
    "Correct predictions:",
    int(field_df["correct"].sum())
)

print("\nMISSING PREDICTIONS BY LANGUAGE")

print(
    field_df.groupby("language")["missing_prediction"]
    .sum()
)

print("\nINCORRECT NONEMPTY PREDICTIONS BY LANGUAGE")

print(
    field_df.groupby("language")["incorrect_nonempty"]
    .sum()
)


TOTAL FIELD PREDICTIONS: 270
Missing predictions: 6
Incorrect nonempty predictions: 73
Correct predictions: 191

MISSING PREDICTIONS BY LANGUAGE
language
english    1
hindi      1
kannada    4
Name: missing_prediction, dtype: int64

INCORRECT NONEMPTY PREDICTIONS BY LANGUAGE
language
english     2
hindi      23
kannada    48
Name: incorrect_nonempty, dtype: int64


In [33]:

error_records = []

for result in zero_shot_results:

    fields = CATEGORY_SCHEMAS[result["category"]]

    for field in fields:

        expected = str(
            result["ground_truth"].get(field, "")
        ).strip()

        predicted = str(
            result["prediction"].get(field, "")
        ).strip()

        if expected != predicted:

            error_records.append({
                "document_id": result["document_id"],
                "language": result["language"],
                "category": result["category"],
                "ocr_engine": result["ocr_engine"],
                "field": field,
                "expected": expected,
                "predicted": predicted,
                "error_type": (
                    "missing"
                    if predicted == ""
                    else "incorrect_nonempty"
                ),
                "ocr_text": next(
                    r["ocr_text"]
                    for r in unified_records
                    if r["document_id"] == result["document_id"]
                    and r["ocr_engine"] == result["ocr_engine"]
                    and r["corruption"] == result["corruption"]
                    and r["severity"] == result["severity"]
                )
            })

error_df = pd.DataFrame(error_records)

print("Total incorrect fields:", len(error_df))

print("\nErrors by language:")
print(error_df["language"].value_counts())

print("\nErrors by field:")
print(error_df["field"].value_counts())


Total incorrect fields: 79

Errors by language:
language
kannada    52
hindi      24
english     3
Name: count, dtype: int64

Errors by field:
field
document_id    24
name           16
date           14
grade          10
address         6
institution     4
phone           4
amount          1
Name: count, dtype: int64


In [34]:

kannada_errors = error_df[
    error_df["language"] == "kannada"
]

for _, row in kannada_errors.head(10).iterrows():

    print("=" * 80)
    print("DOCUMENT:", row["document_id"])
    print("CATEGORY:", row["category"])
    print("OCR ENGINE:", row["ocr_engine"])
    print("FIELD:", row["field"])
    print("EXPECTED:", row["expected"])
    print("PREDICTED:", row["predicted"])
    print("ERROR TYPE:", row["error_type"])

    print("\nOCR TEXT:")
    print(row["ocr_text"])


DOCUMENT: kannada_invoice_067
CATEGORY: invoice
OCR ENGINE: tesseract
FIELD: name
EXPECTED: ವಿಕ್ರಮ್ ಮೆನನ್
PREDICTED: ವಿಕ್ರಮ್‌ ಮೆನನ್‌
ERROR TYPE: incorrect_nonempty

OCR TEXT:
ಸರಕುಪಟ್ಟಿ

ಗ್ರಾಹಕರ ಹೆಸರು :. ವಿಕ್ರಮ್‌ ಮೆನನ್‌
ಸರಕುಪಟ್ಟಿ ದಿನಾಂಕ :.. 08/03/2026
ಸರಕುಪಟ್ಟಿ ಸಂಖ್ಯೆ : .1-0067
ವಿಳಾಸ :. ತುಮಕೂರು

ಒಟ್ಟು ಮೊತ್ತ :.. 10679
DOCUMENT: kannada_invoice_067
CATEGORY: invoice
OCR ENGINE: tesseract
FIELD: document_id
EXPECTED: INV-0067
PREDICTED: 1-0067
ERROR TYPE: incorrect_nonempty

OCR TEXT:
ಸರಕುಪಟ್ಟಿ

ಗ್ರಾಹಕರ ಹೆಸರು :. ವಿಕ್ರಮ್‌ ಮೆನನ್‌
ಸರಕುಪಟ್ಟಿ ದಿನಾಂಕ :.. 08/03/2026
ಸರಕುಪಟ್ಟಿ ಸಂಖ್ಯೆ : .1-0067
ವಿಳಾಸ :. ತುಮಕೂರು

ಒಟ್ಟು ಮೊತ್ತ :.. 10679
DOCUMENT: kannada_invoice_068
CATEGORY: invoice
OCR ENGINE: tesseract
FIELD: name
EXPECTED: ಅನನ್ಯಾ ಕುಮಾರ್
PREDICTED: ಅನನ್ಯಾ ಕುಮಾರ್‌
ERROR TYPE: incorrect_nonempty

OCR TEXT:
ಗ್ರಾಹಕರ ಹೆಸರು :. ಅನನ್ಯಾ ಕುಮಾರ್‌
ಸರಕುಪಟ್ಟಿ ದಿನಾಂಕ :.. 09/03/2026
ಸರಕುಪಟ್ಟಿ ಸಂಖ್ಯೆ ು.. 11/-0068
ವಿಳಾಸ : . ಉಡುಪಿ

ಒಟ್ಟು ಮೊತ್ತ :.. 10816
DOCUMENT: kannada_invoice_068
CATEGORY: invoice
OCR ENGINE: tess

In [35]:

ZERO_SHOT_RESULTS_PATH = os.path.join(
    OUTPUT_DIR,
    "qwen3_zero_shot_validation_results.jsonl"
)

with open(ZERO_SHOT_RESULTS_PATH, "w", encoding="utf-8") as f:

    for result in zero_shot_results:
        f.write(
            json.dumps(result, ensure_ascii=False) + "\n"
        )

ERROR_REPORT_PATH = os.path.join(
    METRICS_DIR,
    "qwen3_zero_shot_errors.csv"
)

error_df.to_csv(
    ERROR_REPORT_PATH,
    index=False,
    encoding="utf-8"
)

print("Zero-shot results saved:")
print(ZERO_SHOT_RESULTS_PATH)

print("\nError report saved:")
print(ERROR_REPORT_PATH)


Zero-shot results saved:
/content/drive/MyDrive/BharatDocAI/experiments/baselines/llm_extraction/qwen3_zero_shot_validation_results.jsonl

Error report saved:
/content/drive/MyDrive/BharatDocAI/outputs/metrics/llm_extraction/qwen3_zero_shot_errors.csv


In [36]:

import unicodedata
import re

def normalize_for_evaluation(value):

    value = str(value)

    # Canonical Unicode normalization.
    value = unicodedata.normalize("NFC", value)

    # Remove zero-width formatting characters.
    value = value.replace("\u200c", "")
    value = value.replace("\u200d", "")

    # Normalize whitespace.
    value = re.sub(r"\s+", " ", value).strip()

    return value


field_df["normalized_expected"] = (
    field_df["expected"].apply(normalize_for_evaluation)
)

field_df["normalized_predicted"] = (
    field_df["predicted"].apply(normalize_for_evaluation)
)

field_df["normalized_correct"] = (
    field_df["normalized_expected"]
    == field_df["normalized_predicted"]
)

raw_accuracy = field_df["correct"].mean() * 100

normalized_accuracy = (
    field_df["normalized_correct"].mean() * 100
)

print("RAW FIELD ACCURACY:", round(raw_accuracy, 2), "%")

print(
    "UNICODE-NORMALIZED FIELD ACCURACY:",
    round(normalized_accuracy, 2),
    "%"
)

print(
    "Formatting-only mismatches corrected:",
    int(
        (
            field_df["normalized_correct"]
            & ~field_df["correct"]
        ).sum()
    )
)


RAW FIELD ACCURACY: 70.74 %
UNICODE-NORMALIZED FIELD ACCURACY: 72.96 %
Formatting-only mismatches corrected: 6


In [37]:

def normalize_ocr_for_matching(value):

    value = unicodedata.normalize("NFC", str(value))

    value = value.replace("\u200c", "")
    value = value.replace("\u200d", "")

    # Normalize whitespace and case.
    value = re.sub(r"\s+", " ", value).strip()

    return value.casefold()


recoverability_results = []

for result in zero_shot_results:

    ocr_text = normalize_ocr_for_matching(
        next(
            r["ocr_text"]
            for r in unified_records
            if r["document_id"] == result["document_id"]
            and r["ocr_engine"] == result["ocr_engine"]
            and r["corruption"] == result["corruption"]
            and r["severity"] == result["severity"]
        )
    )

    for field in CATEGORY_SCHEMAS[result["category"]]:

        expected = str(
            result["ground_truth"].get(field, "")
        ).strip()

        normalized_expected = normalize_ocr_for_matching(expected)

        # Exact normalized substring matching.
        exact_present = (
            normalized_expected in ocr_text
            if normalized_expected else False
        )

        recoverability_results.append({
            "document_id": result["document_id"],
            "language": result["language"],
            "category": result["category"],
            "ocr_engine": result["ocr_engine"],
            "field": field,
            "expected": expected,
            "exact_present_in_ocr": exact_present
        })


recoverability_df = pd.DataFrame(recoverability_results)

print("Total fields examined:", len(recoverability_df))

print(
    "\nExact field value present in OCR:",
    recoverability_df["exact_present_in_ocr"].sum()
)

print(
    "Exact field value absent from OCR:",
    (~recoverability_df["exact_present_in_ocr"]).sum()
)

print(
    "\nOCR exact recoverability percentage:",
    round(
        recoverability_df["exact_present_in_ocr"].mean() * 100,
        2
    ),
    "%"
)


Total fields examined: 270

Exact field value present in OCR: 198
Exact field value absent from OCR: 72

OCR exact recoverability percentage: 73.33 %


In [38]:

print("OCR RECOVERABILITY BY LANGUAGE")

print(
    recoverability_df.groupby("language")[
        "exact_present_in_ocr"
    ].mean().mul(100).round(2)
)

print("\nOCR RECOVERABILITY BY ENGINE")

print(
    recoverability_df.groupby("ocr_engine")[
        "exact_present_in_ocr"
    ].mean().mul(100).round(2)
)

print("\nOCR RECOVERABILITY BY LANGUAGE AND ENGINE")

print(
    recoverability_df.pivot_table(
        index="language",
        columns="ocr_engine",
        values="exact_present_in_ocr",
        aggfunc="mean"
    ).mul(100).round(2)
)


OCR RECOVERABILITY BY LANGUAGE
language
english    96.67
hindi      73.33
kannada    50.00
Name: exact_present_in_ocr, dtype: float64

OCR RECOVERABILITY BY ENGINE
ocr_engine
easyocr      77.78
paddleocr    66.67
tesseract    75.56
Name: exact_present_in_ocr, dtype: float64

OCR RECOVERABILITY BY LANGUAGE AND ENGINE
ocr_engine  easyocr  paddleocr  tesseract
language                                 
english      100.00      96.67      93.33
hindi         70.00      90.00      60.00
kannada       63.33      13.33      73.33


In [39]:

# Merge OCR recoverability with LLM field predictions.

comparison_df = field_df.merge(
    recoverability_df,
    on=[
        "document_id",
        "language",
        "category",
        "ocr_engine",
        "field"
    ],
    how="inner"
)

print("Merged records:", len(comparison_df))

print("\nLLM ACCURACY WHEN EXACT VALUE IS PRESENT IN OCR")

present_accuracy = (
    comparison_df[
        comparison_df["exact_present_in_ocr"]
    ]["normalized_correct"].mean() * 100
)

print(round(present_accuracy, 2), "%")


print("\nLLM ACCURACY WHEN EXACT VALUE IS ABSENT FROM OCR")

absent_accuracy = (
    comparison_df[
        ~comparison_df["exact_present_in_ocr"]
    ]["normalized_correct"].mean() * 100
)

print(round(absent_accuracy, 2), "%")


print("\nDETAILED RESULTS BY LANGUAGE")

language_comparison = (
    comparison_df.groupby(
        ["language", "exact_present_in_ocr"]
    )["normalized_correct"]
    .agg(["sum", "count", "mean"])
)

language_comparison["accuracy_percent"] = (
    language_comparison["mean"] * 100
).round(2)

print(
    language_comparison[
        ["sum", "count", "accuracy_percent"]
    ].to_string()
)


Merged records: 270

LLM ACCURACY WHEN EXACT VALUE IS PRESENT IN OCR
99.49 %

LLM ACCURACY WHEN EXACT VALUE IS ABSENT FROM OCR
0.0 %

DETAILED RESULTS BY LANGUAGE
                               sum  count  accuracy_percent
language exact_present_in_ocr                              
english  False                   0      3              0.00
         True                   87     87            100.00
hindi    False                   0     24              0.00
         True                   66     66            100.00
kannada  False                   0     45              0.00
         True                   44     45             97.78


In [40]:

RECOVERABILITY_PATH = os.path.join(
    METRICS_DIR,
    "qwen3_ocr_recoverability_analysis.csv"
)

comparison_df.to_csv(
    RECOVERABILITY_PATH,
    index=False,
    encoding="utf-8"
)

print("Recoverability analysis saved:")
print(RECOVERABILITY_PATH)


Recoverability analysis saved:
/content/drive/MyDrive/BharatDocAI/outputs/metrics/llm_extraction/qwen3_ocr_recoverability_analysis.csv


In [47]:

from difflib import SequenceMatcher
import re
import unicodedata

def normalize_field_value(value, field):

    value = unicodedata.normalize("NFC", str(value))
    value = value.replace("\u200c", "")
    value = value.replace("\u200d", "")
    value = value.strip().casefold()

    if field in ["date", "phone", "amount", "document_id"]:

        # Ignore visual separators, retaining letters and digits.
        value = re.sub(r"[\s/\\\-:.,]", "", value)

    else:
        value = re.sub(r"\s+", " ", value)

    return value


def best_ocr_similarity(expected, ocr_text, field):

    expected = normalize_field_value(expected, field)

    if not expected:
        return 0.0

    lines = [
        line.strip()
        for line in ocr_text.splitlines()
        if line.strip()
    ]

    candidates = lines.copy()

    for i in range(len(lines)):
        for window in range(2, min(5, len(lines) - i + 1)):
            candidates.append(" ".join(lines[i:i + window]))

    best_score = 0.0

    for candidate in candidates:

        candidate = normalize_field_value(candidate, field)

        score = SequenceMatcher(
            None,
            expected,
            candidate
        ).ratio()

        best_score = max(best_score, score)

    return round(best_score, 4)


partial_results = []

for result in zero_shot_results:

    ocr_key = (
        result["document_id"],
        result["ocr_engine"],
        result["corruption"],
        result["severity"]
    )

    ocr_text = ocr_text_lookup[ocr_key]

    for field in CATEGORY_SCHEMAS[result["category"]]:

        expected = str(
            result["ground_truth"].get(field, "")
        ).strip()

        field_key = (
            result["document_id"],
            result["language"],
            result["category"],
            result["ocr_engine"],
            field
        )

        partial_results.append({
            "document_id": result["document_id"],
            "language": result["language"],
            "category": result["category"],
            "ocr_engine": result["ocr_engine"],
            "field": field,
            "expected": expected,
            "exact_present": recoverability_lookup[field_key],
            "similarity": best_ocr_similarity(
                expected,
                ocr_text,
                field
            ),
            "llm_correct": llm_accuracy_lookup[field_key],
            "ocr_text": ocr_text
        })


partial_df = pd.DataFrame(partial_results)

absent_df = partial_df[
    ~partial_df["exact_present"]
].copy()

print("Fields absent by exact matching:", len(absent_df))

print("\nUpdated similarity distribution:")
print(absent_df["similarity"].describe().round(3))

print("\nSimilarity by language:")
print(
    absent_df.groupby("language")["similarity"]
    .mean().mul(100).round(2)
)


Fields absent by exact matching: 72

Updated similarity distribution:
count    72.000
mean      0.628
std       0.287
min       0.000
25%       0.482
50%       0.667
75%       0.889
max       1.000
Name: similarity, dtype: float64

Similarity by language:
language
english    14.45
hindi      64.62
kannada    65.09
Name: similarity, dtype: float64


In [48]:

# Create a lookup for clean validation OCR text.
clean_ocr_lookup = {
    (r["document_id"], r["ocr_engine"]): r["ocr_text"]
    for r in unified_records
    if r["split"] == "validation"
    and r["corruption"] == "clean"
}

# Add OCR text to the existing partial results.
partial_df["ocr_text"] = partial_df.apply(
    lambda row: clean_ocr_lookup.get(
        (row["document_id"], row["ocr_engine"]),
        ""
    ),
    axis=1
)

absent_df = partial_df[
    ~partial_df["exact_present"]
].copy()

print("OCR text column added!")
print("Records:", len(absent_df))
print("Missing OCR text:", (absent_df["ocr_text"] == "").sum())


OCR text column added!
Records: 72
Missing OCR text: 0


In [49]:
print("TOP PARTIAL OCR MATCHES")
print("=" * 80)

for _, row in absent_df.sort_values(
    "similarity",
    ascending=False
).head(15).iterrows():

    print(
        f"\n{row['document_id']} | "
        f"{row['language']} | "
        f"{row['ocr_engine']} | "
        f"{row['field']}"
    )

    print("Expected:", row["expected"])
    print("Similarity:", row["similarity"])
    print("LLM correct:", row["llm_correct"])

    print("OCR text:")
    print(row["ocr_text"])

TOP PARTIAL OCR MATCHES

hindi_certificate_057 | hindi | easyocr | document_id
Expected: CERT-0057
Similarity: 1.0
LLM correct: False
OCR text:
प्रमाण पत्र
प्राप्तकर्ता का नाम
रवि नायर
जारी करने की तारीख
26/02/2026
प्रमाण पत्र संख्या
CERT 0057
संस्थान
कर्नाटक तकनीकी विश्वविद्यालय
श्रेणी
/+

hindi_certificate_058 | hindi | easyocr | document_id
Expected: CERT-0058
Similarity: 1.0
LLM correct: False
OCR text:
प्रमाण पत्र
प्राप्तकर्ता का नाम
विक्रम वर्मा
जारी करने की तारीख
27/02/2026
प्रमाण पत्र संख्या
CERT 0058
संस्थान
शहर शिक्षा बोर्ड
श्रेणी
8+

kannada_application_078 | kannada | easyocr | phone
Expected: 9000010686
Similarity: 1.0
LLM correct: False
OCR text:
ಅರ್ಜಿ ನಮೂನೆ
ಅರ್ಜಿದಾರರ ಹೆಸರು
ಸ್ನೇಹಾ ನಾಯರ್
ಅರ್ಜಿ ದಿನಾಂಕ
19 |03 |2026
ಅರ್ಜಿ
ಸಂಖ್ಯೆ
APP-0078
ಳಾಸ
ಉಡುಪಿ
ದೂರವಾಣಿ ಸಂಖ್ಯೆ
900001 0686

kannada_application_077 | kannada | easyocr | phone
Expected: 9000010549
Similarity: 1.0
LLM correct: False
OCR text:
ಅರ್ಜಿ ನಮೂನೆ
ಅರ್ಜಿದಾರರ ಹೆಸರು
ವಿಕ್ರಮ್ ರೆಡ್ಡಿ
ಅರ್ಜಿ ದಿನಾಂಕ
18 |03 | 2026
ಅರ್ಜಿ
ಸಂಖ್ಯೆ
APP

In [50]:


for _, row in absent_df.sort_values(
    "similarity",
    ascending=False
).head(15).iterrows():

    print("=" * 80)
    print(row["document_id"], row["language"], row["ocr_engine"])
    print("Field:", row["field"])
    print("Expected:", row["expected"])
    print("Similarity:", row["similarity"])
    print("LLM correct:", row["llm_correct"])
    print("OCR text:\n", row["ocr_text"])

hindi_certificate_057 hindi easyocr
Field: document_id
Expected: CERT-0057
Similarity: 1.0
LLM correct: False
OCR text:
 प्रमाण पत्र
प्राप्तकर्ता का नाम
रवि नायर
जारी करने की तारीख
26/02/2026
प्रमाण पत्र संख्या
CERT 0057
संस्थान
कर्नाटक तकनीकी विश्वविद्यालय
श्रेणी
/+
hindi_certificate_058 hindi easyocr
Field: document_id
Expected: CERT-0058
Similarity: 1.0
LLM correct: False
OCR text:
 प्रमाण पत्र
प्राप्तकर्ता का नाम
विक्रम वर्मा
जारी करने की तारीख
27/02/2026
प्रमाण पत्र संख्या
CERT 0058
संस्थान
शहर शिक्षा बोर्ड
श्रेणी
8+
kannada_application_078 kannada easyocr
Field: phone
Expected: 9000010686
Similarity: 1.0
LLM correct: False
OCR text:
 ಅರ್ಜಿ ನಮೂನೆ
ಅರ್ಜಿದಾರರ ಹೆಸರು
ಸ್ನೇಹಾ ನಾಯರ್
ಅರ್ಜಿ ದಿನಾಂಕ
19 |03 |2026
ಅರ್ಜಿ
ಸಂಖ್ಯೆ
APP-0078
ಳಾಸ
ಉಡುಪಿ
ದೂರವಾಣಿ ಸಂಖ್ಯೆ
900001 0686
kannada_application_077 kannada easyocr
Field: phone
Expected: 9000010549
Similarity: 1.0
LLM correct: False
OCR text:
 ಅರ್ಜಿ ನಮೂನೆ
ಅರ್ಜಿದಾರರ ಹೆಸರು
ವಿಕ್ರಮ್ ರೆಡ್ಡಿ
ಅರ್ಜಿ ದಿನಾಂಕ
18 |03 | 2026
ಅರ್ಜಿ
ಸಂಖ್ಯೆ
APP-0077
ಳಾಸ
ತುಮಕೂರು
ದೂ

In [51]:

# Create a lookup for original Qwen3 predictions.

prediction_lookup = {
    (
        r["document_id"],
        r["ocr_engine"],
        r["category"]
    ): r["prediction"]
    for r in zero_shot_results
}

# Inspect the 15 highest-similarity fields that failed
# the original exact OCR matching.

top_cases = absent_df.sort_values(
    "similarity",
    ascending=False
).head(15)

for _, row in top_cases.iterrows():

    key = (
        row["document_id"],
        row["ocr_engine"],
        row["category"]
    )

    prediction = prediction_lookup[key]

    print("=" * 85)
    print("DOCUMENT:", row["document_id"])
    print("LANGUAGE:", row["language"])
    print("ENGINE:", row["ocr_engine"])
    print("FIELD:", row["field"])

    print("\nEXPECTED:")
    print(row["expected"])

    print("\nQWEN3 PREDICTED:")
    print(prediction.get(row["field"], ""))

    print("\nOCR SIMILARITY:")
    print(row["similarity"])

    print("\nOCR TEXT:")
    print(row["ocr_text"])


DOCUMENT: hindi_certificate_057
LANGUAGE: hindi
ENGINE: easyocr
FIELD: document_id

EXPECTED:
CERT-0057

QWEN3 PREDICTED:
CERT 0057

OCR SIMILARITY:
1.0

OCR TEXT:
प्रमाण पत्र
प्राप्तकर्ता का नाम
रवि नायर
जारी करने की तारीख
26/02/2026
प्रमाण पत्र संख्या
CERT 0057
संस्थान
कर्नाटक तकनीकी विश्वविद्यालय
श्रेणी
/+
DOCUMENT: hindi_certificate_058
LANGUAGE: hindi
ENGINE: easyocr
FIELD: document_id

EXPECTED:
CERT-0058

QWEN3 PREDICTED:
CERT 0058

OCR SIMILARITY:
1.0

OCR TEXT:
प्रमाण पत्र
प्राप्तकर्ता का नाम
विक्रम वर्मा
जारी करने की तारीख
27/02/2026
प्रमाण पत्र संख्या
CERT 0058
संस्थान
शहर शिक्षा बोर्ड
श्रेणी
8+
DOCUMENT: kannada_application_078
LANGUAGE: kannada
ENGINE: easyocr
FIELD: phone

EXPECTED:
9000010686

QWEN3 PREDICTED:
900001 0686

OCR SIMILARITY:
1.0

OCR TEXT:
ಅರ್ಜಿ ನಮೂನೆ
ಅರ್ಜಿದಾರರ ಹೆಸರು
ಸ್ನೇಹಾ ನಾಯರ್
ಅರ್ಜಿ ದಿನಾಂಕ
19 |03 |2026
ಅರ್ಜಿ
ಸಂಖ್ಯೆ
APP-0078
ಳಾಸ
ಉಡುಪಿ
ದೂರವಾಣಿ ಸಂಖ್ಯೆ
900001 0686
DOCUMENT: kannada_application_077
LANGUAGE: kannada
ENGINE: easyocr
FIELD: phone

EXPECTED:
900

In [52]:

reviewed_cases = {
    ("kannada_certificate_088", "paddleocr", "date"),
    ("kannada_invoice_067", "paddleocr", "amount")
}

remaining_kannada_errors = absent_df[
    absent_df["language"] == "kannada"
].copy()

remaining_kannada_errors = remaining_kannada_errors[
    ~remaining_kannada_errors.apply(
        lambda row: (
            row["document_id"],
            row["ocr_engine"],
            row["field"]
        ) in reviewed_cases,
        axis=1
    )
]

for _, row in remaining_kannada_errors.sort_values(
    "similarity",
    ascending=False
).head(15).iterrows():

    key = (
        row["document_id"],
        row["ocr_engine"],
        row["category"]
    )

    prediction = prediction_lookup[key]

    print("=" * 80)
    print("DOCUMENT:", row["document_id"])
    print("ENGINE:", row["ocr_engine"])
    print("FIELD:", row["field"])
    print("EXPECTED:", row["expected"])
    print("PREDICTED:", prediction.get(row["field"], ""))
    print("SIMILARITY:", row["similarity"])
    print("\nOCR TEXT:")
    print(row["ocr_text"])


DOCUMENT: kannada_application_077
ENGINE: easyocr
FIELD: phone
EXPECTED: 9000010549
PREDICTED: 900001 0549
SIMILARITY: 1.0

OCR TEXT:
ಅರ್ಜಿ ನಮೂನೆ
ಅರ್ಜಿದಾರರ ಹೆಸರು
ವಿಕ್ರಮ್ ರೆಡ್ಡಿ
ಅರ್ಜಿ ದಿನಾಂಕ
18 |03 | 2026
ಅರ್ಜಿ
ಸಂಖ್ಯೆ
APP-0077
ಳಾಸ
ತುಮಕೂರು
ದೂರವಾಣಿ ಸಂಖ್ಯೆ
900001 0549
DOCUMENT: kannada_application_078
ENGINE: easyocr
FIELD: phone
EXPECTED: 9000010686
PREDICTED: 900001 0686
SIMILARITY: 1.0

OCR TEXT:
ಅರ್ಜಿ ನಮೂನೆ
ಅರ್ಜಿದಾರರ ಹೆಸರು
ಸ್ನೇಹಾ ನಾಯರ್
ಅರ್ಜಿ ದಿನಾಂಕ
19 |03 |2026
ಅರ್ಜಿ
ಸಂಖ್ಯೆ
APP-0078
ಳಾಸ
ಉಡುಪಿ
ದೂರವಾಣಿ ಸಂಖ್ಯೆ
900001 0686
DOCUMENT: kannada_application_078
ENGINE: paddleocr
FIELD: phone
EXPECTED: 9000010686
PREDICTED: 90000106ಢ6
SIMILARITY: 0.9

OCR TEXT:
ಅಜಿ೯ನಮೂನ
ಅಜಿ೯ದಾರರಹೆಸರು
ಸೀಹಾನಾಯರ್
ಅಜಿ೯ದನಾಂಕ
19 10312026
ಅಜಿ೯ಸಂಖೇ
APP-OO78
ಎಳಾಸ
ಉಡುಏ
ದೂರವಾಣಿಸಂಖ
90000106ಢ6
DOCUMENT: kannada_invoice_067
ENGINE: easyocr
FIELD: date
EXPECTED: 08/03/2026
PREDICTED: 08 |03 | 2026
SIMILARITY: 0.8889

OCR TEXT:
ಸರಕುಪಟ್ಟಿ
ಗ್ರಾಹಕರ ಹೆಸರು
ವಿಕ್ರಮ್ ಮೆನನ್
ಸರಕುಪಟ್ಟಿ ದಿನಾಂಕ
08 |03 | 2026
ಸರಕುಪಟ್ಟಿ ಸಂಖ್ಯೆ
[NV-0067
ವಿಳಾಸ


In [53]:

import re
import unicodedata

def normalize_field_value(value, field):
    value = unicodedata.normalize("NFC", str(value))

    value = value.replace("\u200c", "")
    value = value.replace("\u200d", "")

    value = value.strip().casefold()

    if field in ["date", "phone", "amount", "document_id"]:
        # Remove formatting separators, but do not change characters.
        value = re.sub(r"[\s/\\\-:.,|]", "", value)

    else:
        value = re.sub(r"\s+", " ", value)

    return value


# Verify the examples
checks = [
    ("9000010549", "900001 0549", "phone"),
    ("08/03/2026", "08 |03 | 2026", "date"),
    ("INV-0067", "[NV-0067", "document_id"),
    ("10679", "1067", "amount")
]

for expected, predicted, field in checks:
    match = (
        normalize_field_value(expected, field)
        == normalize_field_value(predicted, field)
    )

    print(
        field,
        "| Expected:", expected,
        "| Predicted:", predicted,
        "| Correct:", match
    )


phone | Expected: 9000010549 | Predicted: 900001 0549 | Correct: True
date | Expected: 08/03/2026 | Predicted: 08 |03 | 2026 | Correct: True
document_id | Expected: INV-0067 | Predicted: [NV-0067 | Correct: False
amount | Expected: 10679 | Predicted: 1067 | Correct: False


In [54]:

# Recalculate correctness using field-aware normalization

normalized_correct = []

for _, row in field_df.iterrows():
    expected = row["expected"]
    predicted = row["predicted"]
    field = row["field"]

    correct = (
        normalize_field_value(expected, field)
        == normalize_field_value(predicted, field)
    )

    normalized_correct.append(correct)

field_df["normalized_correct"] = normalized_correct

raw_accuracy = field_df["correct"].mean() * 100
normalized_accuracy = field_df["normalized_correct"].mean() * 100

print(f"Raw field accuracy: {raw_accuracy:.2f}%")
print(f"Normalized field accuracy: {normalized_accuracy:.2f}%")

print("\nAccuracy by language:")

print(
    field_df.groupby("language")[
        ["correct", "normalized_correct"]
    ].mean().mul(100).round(2)
)

print("\nAccuracy by OCR engine:")

print(
    field_df.groupby("ocr_engine")[
        ["correct", "normalized_correct"]
    ].mean().mul(100).round(2)
)


Raw field accuracy: 70.74%
Normalized field accuracy: 76.67%

Accuracy by language:
          correct  normalized_correct
language                             
english     96.67               96.67
hindi       73.33               75.56
kannada     42.22               57.78

Accuracy by OCR engine:
            correct  normalized_correct
ocr_engine                             
easyocr       77.78               88.89
paddleocr     66.67               66.67
tesseract     67.78               74.44


In [55]:

normalized_doc_results = []

for (document_id, language, engine), group in field_df.groupby(
    ["document_id", "language", "ocr_engine"]
):
    all_fields_correct = group["normalized_correct"].all()

    normalized_doc_results.append({
        "document_id": document_id,
        "language": language,
        "ocr_engine": engine,
        "exact_match": all_fields_correct
    })

normalized_doc_df = pd.DataFrame(normalized_doc_results)

print("Total document-engine evaluations:", len(normalized_doc_df))

print(
    "Normalized exact document match:",
    round(normalized_doc_df["exact_match"].mean() * 100, 2),
    "%"
)

print("\nBy language:")
print(
    normalized_doc_df.groupby("language")["exact_match"]
    .mean().mul(100).round(2)
)

print("\nBy OCR engine:")
print(
    normalized_doc_df.groupby("ocr_engine")["exact_match"]
    .mean().mul(100).round(2)
)


Total document-engine evaluations: 54
Normalized exact document match: 42.59 %

By language:
language
english    83.33
hindi      22.22
kannada    22.22
Name: exact_match, dtype: float64

By OCR engine:
ocr_engine
easyocr      61.11
paddleocr    44.44
tesseract    22.22
Name: exact_match, dtype: float64


In [56]:

import json
import os

baseline_summary = {
    "model": "Qwen/Qwen3-4B",
    "experiment": "zero_shot",
    "dataset": {
        "validation_documents": 18,
        "ocr_engines": 3,
        "document_engine_evaluations": 54
    },
    "normalized_field_accuracy": 76.67,
    "raw_field_accuracy": 70.74,
    "normalized_exact_document_match": 42.59,
    "exact_document_match_by_language": {
        "english": 83.33,
        "hindi": 22.22,
        "kannada": 22.22
    },
    "exact_document_match_by_engine": {
        "easyocr": 61.11,
        "paddleocr": 44.44,
        "tesseract": 22.22
    }
}

output_path = os.path.join(
    METRICS_DIR,
    "qwen3_zero_shot_baseline_summary.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(baseline_summary, f, ensure_ascii=False, indent=4)

print("Saved:", output_path)


Saved: /content/drive/MyDrive/BharatDocAI/outputs/metrics/llm_extraction/qwen3_zero_shot_baseline_summary.json


In [57]:

print("Unified record keys:")
print(unified_records[0].keys())

print("\nSample training records:")

for record in unified_records:
    if record["split"] == "train" and record["corruption"] == "clean":
        print("\nLanguage:", record["language"])
        print("Category:", record["category"])
        print("OCR engine:", record.get("ocr_engine", record.get("engine")))
        print("OCR text:", record["ocr_text"][:250])
        print("Ground truth:", record["ground_truth"])
        break


Unified record keys:
dict_keys(['document_id', 'language', 'category', 'template_family', 'split', 'ocr_engine', 'corruption', 'severity', 'ocr_text', 'ground_truth', 'ocr_status'])

Sample training records:

Language: english
Category: invoice
OCR engine: tesseract
OCR text: INVOICE

Customer Name:

Invoice Date:

Invoice Number:

Address:

Total Amount:

Kiran Rao

01/01/2026

INV-0001

Bengaluru

1637
Ground truth: {'name': 'Kiran Rao', 'date': '01/01/2026', 'document_id': 'INV-0001', 'address': 'Bengaluru', 'amount': '1637'}


In [58]:

from collections import defaultdict
import random

random.seed(42)

# Only clean training records can be demonstrations
training_examples = [
    record for record in unified_records
    if record["split"] == "train"
    and record["corruption"] == "clean"
    and record["ocr_status"] == "success"
]

# Group by language, category and OCR engine
demonstration_pool = defaultdict(list)

for record in training_examples:
    key = (
        record["language"],
        record["category"],
        record["ocr_engine"]
    )
    demonstration_pool[key].append(record)

# Check coverage
print("Total clean training OCR records:", len(training_examples))
print("Available demonstration groups:", len(demonstration_pool))

for key in sorted(demonstration_pool):
    print(key, ":", len(demonstration_pool[key]), "examples")

# Verify all 27 combinations exist
languages = ["english", "hindi", "kannada"]
categories = ["invoice", "application", "certificate"]
engines = ["tesseract", "easyocr", "paddleocr"]

missing_groups = [
    (language, category, engine)
    for language in languages
    for category in categories
    for engine in engines
    if (language, category, engine) not in demonstration_pool
]

print("\nMissing groups:", missing_groups)


Total clean training OCR records: 162
Available demonstration groups: 27
('english', 'application', 'easyocr') : 6 examples
('english', 'application', 'paddleocr') : 6 examples
('english', 'application', 'tesseract') : 6 examples
('english', 'certificate', 'easyocr') : 6 examples
('english', 'certificate', 'paddleocr') : 6 examples
('english', 'certificate', 'tesseract') : 6 examples
('english', 'invoice', 'easyocr') : 6 examples
('english', 'invoice', 'paddleocr') : 6 examples
('english', 'invoice', 'tesseract') : 6 examples
('hindi', 'application', 'easyocr') : 6 examples
('hindi', 'application', 'paddleocr') : 6 examples
('hindi', 'application', 'tesseract') : 6 examples
('hindi', 'certificate', 'easyocr') : 6 examples
('hindi', 'certificate', 'paddleocr') : 6 examples
('hindi', 'certificate', 'tesseract') : 6 examples
('hindi', 'invoice', 'easyocr') : 6 examples
('hindi', 'invoice', 'paddleocr') : 6 examples
('hindi', 'invoice', 'tesseract') : 6 examples
('kannada', 'application', 

In [69]:

few_shot_examples = {}
demonstration_quality = []

languages = ["english", "hindi", "kannada"]
categories = ["invoice", "application", "certificate"]

for language in languages:
    for category in categories:

        candidates = []

        for engine in ["easyocr", "tesseract", "paddleocr"]:
            key = (language, category, engine)

            for record in demonstration_pool.get(key, []):

                gt = record["ground_truth"]
                ocr_text = record["ocr_text"]

                supported_fields = []

                for field, expected in gt.items():
                    expected_norm = normalize_field_value(expected, field)
                    ocr_norm = normalize_field_value(ocr_text, field)

                    supported = (
                        expected_norm != ""
                        and expected_norm in ocr_norm
                    )

                    supported_fields.append(supported)

                candidates.append({
                    "record": record,
                    "score": sum(supported_fields),
                    "total": len(supported_fields)
                })

        # Prefer maximum OCR-supported fields.
        # Use deterministic tie-breaking.
        engine_priority = {
            "easyocr": 0,
            "tesseract": 1,
            "paddleocr": 2
        }

        candidates.sort(
            key=lambda x: (
                -x["score"],
                engine_priority[x["record"]["ocr_engine"]],
                x["record"]["document_id"]
            )
        )

        best = candidates[0]
        selected = best["record"]

        few_shot_examples[(language, category)] = selected

        demonstration_quality.append({
            "language": language,
            "category": category,
            "source_engine": selected["ocr_engine"],
            "document_id": selected["document_id"],
            "supported_fields": best["score"],
            "total_fields": best["total"]
        })

quality_df = pd.DataFrame(demonstration_quality)

print("Selected demonstrations:", len(few_shot_examples))
display(quality_df)

print(
    "\nFully supported demonstrations:",
    (quality_df["supported_fields"] == 5).sum(),
    "/ 9"
)


Selected demonstrations: 9


,language,category,source_engine,document_id,supported_fields,total_fields
0,english,invoice,easyocr,english_invoice_001,5,5
1,english,application,easyocr,english_application_011,5,5
2,english,certificate,easyocr,english_certificate_021,5,5
3,hindi,invoice,paddleocr,hindi_invoice_033,5,5
4,hindi,application,easyocr,hindi_application_041,5,5
5,hindi,certificate,paddleocr,hindi_certificate_052,5,5
6,kannada,invoice,easyocr,kannada_invoice_061,4,5
7,kannada,application,easyocr,kannada_application_071,5,5
8,kannada,certificate,easyocr,kannada_certificate_083,5,5



Fully supported demonstrations: 8 / 9


In [72]:

def build_few_shot_prompt(record):

    key = (
        record["language"],
        record["category"]
    )

    example = few_shot_examples[key]

    # Only provide ground-truth values supported by the example OCR.
    supported_ground_truth = {}

    for field, expected in example["ground_truth"].items():
        expected_norm = normalize_field_value(expected, field)
        ocr_norm = normalize_field_value(example["ocr_text"], field)

        if expected_norm and expected_norm in ocr_norm:
            supported_ground_truth[field] = expected
        else:
            supported_ground_truth[field] = ""

    example_json = json.dumps(
        supported_ground_truth,
        ensure_ascii=False,
        indent=2
    )

    demonstration = f"""
Example OCR text:
{example["ocr_text"]}

Correct JSON output:
{example_json}

Use this example to understand the extraction format.
Do not copy values from the example.
Extract only information supported by the current document.
"""

    return demonstration + "\n\n" + build_extraction_prompt(record)


In [76]:

key = ("kannada", "invoice")

for engine in ["easyocr", "tesseract", "paddleocr"]:
    print("\nENGINE:", engine)

    for record in demonstration_pool.get(
        ("kannada", "invoice", engine), []
    ):
        gt = record["ground_truth"]
        ocr_text = record["ocr_text"]

        supported = []

        for field, expected in gt.items():
            expected_norm = normalize_field_value(expected, field)
            ocr_norm = normalize_field_value(ocr_text, field)

            supported.append(
                expected_norm != "" and expected_norm in ocr_norm
            )

        print(
            record["document_id"],
            "|",
            sum(supported),
            "/",
            len(supported)
        )



ENGINE: easyocr
kannada_invoice_061 | 4 / 5
kannada_invoice_062 | 4 / 5
kannada_invoice_063 | 4 / 5
kannada_invoice_064 | 3 / 5
kannada_invoice_065 | 4 / 5
kannada_invoice_066 | 3 / 5

ENGINE: tesseract
kannada_invoice_061 | 4 / 5
kannada_invoice_062 | 4 / 5
kannada_invoice_063 | 4 / 5
kannada_invoice_064 | 4 / 5
kannada_invoice_065 | 4 / 5
kannada_invoice_066 | 4 / 5

ENGINE: paddleocr
kannada_invoice_061 | 1 / 5
kannada_invoice_062 | 0 / 5
kannada_invoice_063 | 2 / 5
kannada_invoice_064 | 0 / 5
kannada_invoice_065 | 1 / 5
kannada_invoice_066 | 1 / 5


In [77]:

for engine in ["easyocr", "tesseract"]:
    record = next(
        r for r in demonstration_pool[
            ("kannada", "invoice", engine)
        ]
        if r["document_id"] == "kannada_invoice_064"
    ) if engine == "tesseract" else next(
        r for r in demonstration_pool[
            ("kannada", "invoice", engine)
        ]
        if r["document_id"] == "kannada_invoice_061"
    )

    print("\nENGINE:", engine)
    print("DOCUMENT:", record["document_id"])

    for field, expected in record["ground_truth"].items():
        expected_norm = normalize_field_value(expected, field)
        ocr_norm = normalize_field_value(record["ocr_text"], field)

        supported = (
            expected_norm != ""
            and expected_norm in ocr_norm
        )

        print(
            field,
            "| Supported:", supported,
            "| Expected:", expected
        )

    print("\nOCR TEXT:")
    print(record["ocr_text"])



ENGINE: easyocr
DOCUMENT: kannada_invoice_061
name | Supported: True | Expected: ಕಿರಣ್ ಮೆನನ್
date | Supported: True | Expected: 02/03/2026
document_id | Supported: False | Expected: INV-0061
address | Supported: True | Expected: ಬೆಂಗಳೂರು
amount | Supported: True | Expected: 9857

OCR TEXT:
ಸರಕುಪಟ್ಟಿ
ಹೆಸರು:
ಕಿರಣ್ ಮೆನನ್
ಸರಕುಪಟ್ಟಿ ದಿನಾಂಕ:
02 |03 | 2026
ಸರಕುಪಟ್ಟಿ ಸಂಖ್ಯೆ:
[NV-0061
ವಿಳಾಸ:
ಬೆಂಗಳೂರು
ಒಟ್ಚು ಮೊತ್ತ:
9857
ಗ್ರಾಹಕರ

ENGINE: tesseract
DOCUMENT: kannada_invoice_064
name | Supported: True | Expected: ಅನನ್ಯಾ ಮೆನನ್
date | Supported: True | Expected: 05/03/2026
document_id | Supported: False | Expected: INV-0064
address | Supported: True | Expected: ಹುಬ್ಬಳ್ಳಿ
amount | Supported: True | Expected: 10268

OCR TEXT:
ಸರಕುಪಟ್ಟಿ

ಗ್ರಾಹಕರ ಹೆಸರು ಅನನ್ಯಾ ಮೆನನ್‌
ಸರಕುಪಟ್ಟಿ ದಿನಾಂಕ 05/03/2026
ಸರಕುಪಟ್ಟಿ ಸಂಖ್ಯೆ 1/-0064
ವಿಳಾಸ ಹುಬ್ಬಳ್ಳಿ

ಒಟ್ಟು ಮೊತ್ತ 10268


In [78]:

# Replace the Kannada invoice demonstration
selected_record = next(
    r for r in demonstration_pool[
        ("kannada", "invoice", "tesseract")
    ]
    if r["document_id"] == "kannada_invoice_064"
)

few_shot_examples[("kannada", "invoice")] = selected_record

# Verify all nine demonstrations
print("Final demonstration count:", len(few_shot_examples))

for key, record in few_shot_examples.items():
    print(
        key,
        "|",
        record["document_id"],
        "|",
        record["ocr_engine"]
    )

print("\nSelected Kannada invoice:")
print(selected_record["ocr_text"])


Final demonstration count: 9
('english', 'invoice') | english_invoice_001 | easyocr
('english', 'application') | english_application_011 | easyocr
('english', 'certificate') | english_certificate_021 | easyocr
('hindi', 'invoice') | hindi_invoice_033 | paddleocr
('hindi', 'application') | hindi_application_041 | easyocr
('hindi', 'certificate') | hindi_certificate_052 | paddleocr
('kannada', 'invoice') | kannada_invoice_064 | tesseract
('kannada', 'application') | kannada_application_071 | easyocr
('kannada', 'certificate') | kannada_certificate_083 | easyocr

Selected Kannada invoice:
ಸರಕುಪಟ್ಟಿ

ಗ್ರಾಹಕರ ಹೆಸರು ಅನನ್ಯಾ ಮೆನನ್‌
ಸರಕುಪಟ್ಟಿ ದಿನಾಂಕ 05/03/2026
ಸರಕುಪಟ್ಟಿ ಸಂಖ್ಯೆ 1/-0064
ವಿಳಾಸ ಹುಬ್ಬಳ್ಳಿ

ಒಟ್ಟು ಮೊತ್ತ 10268


In [80]:

def build_few_shot_prompt(record):

    # Select demonstration based on language and document category
    key = (
        record["language"],
        record["category"]
    )

    example = few_shot_examples[key]

    # Build demonstration ground truth.
    # Only include values supported by the demonstration OCR.
    supported_ground_truth = {}

    for field, expected in example["ground_truth"].items():

        expected_norm = normalize_field_value(
            expected, field
        )

        ocr_norm = normalize_field_value(
            example["ocr_text"], field
        )

        supported = (
            expected_norm != ""
            and expected_norm in ocr_norm
        )

        if supported:
            supported_ground_truth[field] = expected
        else:
            supported_ground_truth[field] = ""

    # Convert demonstration output to JSON
    example_json = json.dumps(
        supported_ground_truth,
        ensure_ascii=False,
        indent=2
    )

    # Construct demonstration section
    demonstration = f"""
You are given an example of structured information extraction.

EXAMPLE OCR TEXT:
{example["ocr_text"]}

CORRECT JSON OUTPUT:
{example_json}

Learn the extraction format from this example.
Do not copy values from the example.
Extract information only from the current document.
If information is missing or unreadable, return an empty string.
"""

    # Reuse the existing zero-shot extraction instructions
    original_prompt = build_extraction_prompt(record)

    return demonstration + "\n\n" + original_prompt


# Recreate clean validation records if needed
validation_records = [
    r for r in unified_records
    if r["split"] == "validation"
    and r["corruption"] == "clean"
    and r["ocr_status"] == "success"
]

print("Validation records:", len(validation_records))


# Inspect a Kannada invoice validation prompt
sample_validation = next(
    r for r in validation_records
    if r["language"] == "kannada"
    and r["category"] == "invoice"
    and r["ocr_engine"] == "easyocr"
)

sample_prompt = build_few_shot_prompt(sample_validation)

print("\nSelected demonstration:")
print(
    few_shot_examples[
        ("kannada", "invoice")
    ]["document_id"]
)

print("\nGenerated few-shot prompt:\n")
print(sample_prompt)


Validation records: 54

Selected demonstration:
kannada_invoice_064

Generated few-shot prompt:


You are given an example of structured information extraction.

EXAMPLE OCR TEXT:
ಸರಕುಪಟ್ಟಿ

ಗ್ರಾಹಕರ ಹೆಸರು ಅನನ್ಯಾ ಮೆನನ್‌
ಸರಕುಪಟ್ಟಿ ದಿನಾಂಕ 05/03/2026
ಸರಕುಪಟ್ಟಿ ಸಂಖ್ಯೆ 1/-0064
ವಿಳಾಸ ಹುಬ್ಬಳ್ಳಿ

ಒಟ್ಟು ಮೊತ್ತ 10268

CORRECT JSON OUTPUT:
{
  "name": "ಅನನ್ಯಾ ಮೆನನ್",
  "date": "05/03/2026",
  "document_id": "",
  "address": "ಹುಬ್ಬಳ್ಳಿ",
  "amount": "10268"
}

Learn the extraction format from this example.
Do not copy values from the example.
Extract information only from the current document.
If information is missing or unreadable, return an empty string.


You are an information extraction system.

Extract structured information from noisy OCR text.

Document category: invoice
Document language: kannada


Extract these fields:
name, date, document_id, address, amount.
Return exactly these five fields.


Rules:
1. Extract only information explicitly supported by the OCR text.
2. Never invent or gu

In [81]:

def extract_with_qwen_few_shot(record):

    prompt = build_few_shot_prompt(record)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a precise structured information "
                "extraction system. Return only valid JSON."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()


# Test on one validation record
test_response = extract_with_qwen_few_shot(sample_validation)

print("Document:", sample_validation["document_id"])
print("Language:", sample_validation["language"])
print("Category:", sample_validation["category"])
print("OCR engine:", sample_validation["ocr_engine"])

print("\nFew-shot prediction:")
print(test_response)


Document: kannada_invoice_067
Language: kannada
Category: invoice
OCR engine: easyocr

Few-shot prediction:
{
  "name": "ವಿಕ್ರಮ್ ಮೆನನ್",
  "date": "08 |03 | 2026",
  "document_id": "NV-0067",
  "address": "ತುಮಕೂರು",
  "amount": "10679"
}


In [82]:

few_shot_results = []

few_shot_output_path = os.path.join(
    OUTPUT_DIR,
    "qwen3_few_shot_validation_results.jsonl"
)

for i, record in enumerate(
    tqdm(validation_records, desc="Few-shot inference")
):

    raw_response = extract_with_qwen_few_shot(record)

    result = {
        "document_id": record["document_id"],
        "language": record["language"],
        "category": record["category"],
        "ocr_engine": record["ocr_engine"],
        "ground_truth": record["ground_truth"],
        "raw_response": raw_response
    }

    few_shot_results.append(result)

    # Save progress every 5 records
    if (i + 1) % 5 == 0 or (i + 1) == len(validation_records):

        with open(
            few_shot_output_path,
            "w",
            encoding="utf-8"
        ) as f:

            for item in few_shot_results:
                f.write(
                    json.dumps(item, ensure_ascii=False) + "\n"
                )

print("\nFew-shot inference completed.")
print("Total predictions:", len(few_shot_results))
print("Saved to:", few_shot_output_path)


Few-shot inference:   0%|          | 0/54 [00:00<?, ?it/s]


Few-shot inference completed.
Total predictions: 54
Saved to: /content/drive/MyDrive/BharatDocAI/experiments/baselines/llm_extraction/qwen3_few_shot_validation_results.jsonl


In [83]:

parsed_few_shot_results = []

required_fields = {
    "invoice": ["name", "date", "document_id", "address", "amount"],
    "application": ["name", "date", "document_id", "address", "phone"],
    "certificate": ["name", "date", "document_id", "institution", "grade"]
}

for result in few_shot_results:

    raw = result["raw_response"]

    try:
        prediction = json.loads(raw)

        expected_fields = set(
            required_fields[result["category"]]
        )

        actual_fields = set(prediction.keys())

        valid = actual_fields == expected_fields

        parsed_few_shot_results.append({
            **result,
            "prediction": prediction,
            "json_valid": True,
            "schema_valid": valid
        })

    except (json.JSONDecodeError, TypeError):
        parsed_few_shot_results.append({
            **result,
            "prediction": {},
            "json_valid": False,
            "schema_valid": False
        })

print("Total responses:", len(parsed_few_shot_results))
print(
    "Valid JSON:",
    sum(r["json_valid"] for r in parsed_few_shot_results)
)
print(
    "Valid schemas:",
    sum(r["schema_valid"] for r in parsed_few_shot_results)
)


Total responses: 54
Valid JSON: 54
Valid schemas: 54


In [84]:

few_shot_field_rows = []

for result in parsed_few_shot_results:

    prediction = result["prediction"]
    ground_truth = result["ground_truth"]

    for field, expected in ground_truth.items():

        predicted = prediction.get(field, "")

        raw_correct = str(expected) == str(predicted)

        normalized_correct = (
            normalize_field_value(expected, field)
            == normalize_field_value(predicted, field)
        )

        few_shot_field_rows.append({
            "document_id": result["document_id"],
            "language": result["language"],
            "category": result["category"],
            "ocr_engine": result["ocr_engine"],
            "field": field,
            "expected": expected,
            "predicted": predicted,
            "correct": raw_correct,
            "normalized_correct": normalized_correct
        })

few_shot_field_df = pd.DataFrame(few_shot_field_rows)

print("Total fields:", len(few_shot_field_df))

print(
    "Raw field accuracy:",
    round(few_shot_field_df["correct"].mean() * 100, 2),
    "%"
)

print(
    "Normalized field accuracy:",
    round(few_shot_field_df["normalized_correct"].mean() * 100, 2),
    "%"
)

print("\nAccuracy by language:")
print(
    few_shot_field_df.groupby("language")[
        ["correct", "normalized_correct"]
    ].mean().mul(100).round(2)
)

print("\nAccuracy by OCR engine:")
print(
    few_shot_field_df.groupby("ocr_engine")[
        ["correct", "normalized_correct"]
    ].mean().mul(100).round(2)
)


Total fields: 270
Raw field accuracy: 72.96 %
Normalized field accuracy: 77.04 %

Accuracy by language:
          correct  normalized_correct
language                             
english     96.67               96.67
hindi       73.33               75.56
kannada     48.89               58.89

Accuracy by OCR engine:
            correct  normalized_correct
ocr_engine                             
easyocr       82.22               88.89
paddleocr     66.67               66.67
tesseract     70.00               75.56


In [85]:

few_shot_doc_results = []

for (document_id, language, engine), group in few_shot_field_df.groupby(
    ["document_id", "language", "ocr_engine"]
):

    few_shot_doc_results.append({
        "document_id": document_id,
        "language": language,
        "ocr_engine": engine,
        "exact_match": group["normalized_correct"].all()
    })

few_shot_doc_df = pd.DataFrame(few_shot_doc_results)

print(
    "Normalized exact document match:",
    round(few_shot_doc_df["exact_match"].mean() * 100, 2),
    "%"
)

print("\nBy language:")
print(
    few_shot_doc_df.groupby("language")["exact_match"]
    .mean().mul(100).round(2)
)

print("\nBy OCR engine:")
print(
    few_shot_doc_df.groupby("ocr_engine")["exact_match"]
    .mean().mul(100).round(2)
)


Normalized exact document match: 42.59 %

By language:
language
english    83.33
hindi      22.22
kannada    22.22
Name: exact_match, dtype: float64

By OCR engine:
ocr_engine
easyocr      61.11
paddleocr    44.44
tesseract    22.22
Name: exact_match, dtype: float64


In [86]:

comparison = pd.DataFrame({
    "Metric": [
        "Raw field accuracy",
        "Normalized field accuracy",
        "Normalized exact document match"
    ],
    "Zero-shot": [
        field_df["correct"].mean() * 100,
        field_df["normalized_correct"].mean() * 100,
        normalized_doc_df["exact_match"].mean() * 100
    ],
    "Few-shot": [
        few_shot_field_df["correct"].mean() * 100,
        few_shot_field_df["normalized_correct"].mean() * 100,
        few_shot_doc_df["exact_match"].mean() * 100
    ]
})

comparison["Change (percentage points)"] = (
    comparison["Few-shot"] - comparison["Zero-shot"]
)

print(comparison.round(2).to_string(index=False))


                         Metric  Zero-shot  Few-shot  Change (percentage points)
             Raw field accuracy      70.74     72.96                        2.22
      Normalized field accuracy      76.67     77.04                        0.37
Normalized exact document match      42.59     42.59                        0.00


In [87]:

# Save few-shot predictions
few_shot_results_path = os.path.join(
    OUTPUT_DIR,
    "qwen3_few_shot_validation_results.jsonl"
)

with open(few_shot_results_path, "w", encoding="utf-8") as f:
    for result in parsed_few_shot_results:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")


# Save field-level evaluation
few_shot_field_path = os.path.join(
    METRICS_DIR,
    "qwen3_few_shot_field_metrics.csv"
)

few_shot_field_df.to_csv(few_shot_field_path, index=False)


# Save exact document evaluation
few_shot_doc_path = os.path.join(
    METRICS_DIR,
    "qwen3_few_shot_document_metrics.csv"
)

few_shot_doc_df.to_csv(few_shot_doc_path, index=False)


# Save zero-shot vs few-shot comparison
comparison_path = os.path.join(
    METRICS_DIR,
    "zero_shot_vs_few_shot_comparison.csv"
)

comparison.to_csv(comparison_path, index=False)


# Save summary
few_shot_summary = {
    "model": "Qwen/Qwen3-4B",
    "experiment": "few_shot",
    "demonstrations": 9,
    "validation_documents": 18,
    "document_engine_evaluations": 54,
    "total_fields": 270,
    "valid_json": 54,
    "valid_schemas": 54,
    "raw_field_accuracy": round(
        few_shot_field_df["correct"].mean() * 100, 2
    ),
    "normalized_field_accuracy": round(
        few_shot_field_df["normalized_correct"].mean() * 100, 2
    ),
    "normalized_exact_document_match": round(
        few_shot_doc_df["exact_match"].mean() * 100, 2
    )
}

summary_path = os.path.join(
    METRICS_DIR,
    "qwen3_few_shot_summary.json"
)

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(few_shot_summary, f, ensure_ascii=False, indent=4)

print("All few-shot results saved successfully.")
print(summary_path)


All few-shot results saved successfully.
/content/drive/MyDrive/BharatDocAI/outputs/metrics/llm_extraction/qwen3_few_shot_summary.json
